In [2]:
import wandb
import math
import statistics as stats
import numpy as np

In [2]:
w1 = np.array([2E-2, 2E-2, 1E-3, 2E-3, 2E-3, 3E-3, 5E-3, 6E-3])
log_w1 = np.log(w1)
log_avg1 = np.mean(log_w1)
log_mean_w1 = np.exp(log_avg1)  # geometric mean
print(f'{log_mean_w1:.1E}')





w2 = np.array([1E-4, 1E-6, 1E-6, 1E-6, 1E-6, 1E-7, 1E-7, 1E-8])
log_w2 = np.log(w2)
log_avg2 = np.mean(log_w2)
log_mean_w2 = np.exp(log_avg2)  # geometric mean
print(f'{log_mean_w2:.1E}')

4.4E-03
5.6E-07


In [3]:
w1 = np.array([1E-2, 5E-2, 7E-2, 5E-2, 4E-2, 3E-2, 2E-2, 1E-2])
log_w1 = np.log(w1)
log_avg1 = np.mean(log_w1)
log_mean_w1 = np.exp(log_avg1)  # geometric mean
print(f'{log_mean_w1:.1E}')

2.8E-02


In [3]:
# ---- CONFIG ----
entity = "julian-zim-uni-vie"  # e.g. your wandb username or team name
projects = ['hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30',
			'hgm-ll0-05-12-08h57-17-ZIM2165_Gcamp7b_worm1-2022-11-30',
			'hgm-ll0-04-12-11h04-12-2023-12-19_18-27_wt_FM_worm5again2-2023-12-19',
			'hgm-ll0-03-12-20h22-11-2023-12-19_17-35_wt_FM_worm4-2023-12-19',
			'hgm-ll0-03-12-06h26-7-2023-11-21_14-46_wt_worm4-2023-11-21',
			'hgm-ll0-02-12-16h16-1-2022-11-23_worm11',
			'hgm-ll0-02-12-01h36-0-2022-11-23_worm10',
			'hgm-ll0-01-12-11h07-19-ZIM2165_Gcamp7b_worm1-2022_11_28',]

metric_name = "mpe_loss"			 # metric to rank runs by
hyperparam = "w1_reg_lambda"		  # hyperparameter to average
top_fraction = 0.05				  # top 5%



name_prefix = None					   # e.g. "sweep-sweep-" or None to disable

In [ ]:
# ------------- LOGIC -------------
api = wandb.Api()
avg_per_project = {}

for project in projects:
	print(f"\nProcessing project: {project}")

	# Use filters so W&B doesn't send you *everything*:
	# only runs that actually logged the metric.
	filters = {
		f"summary_metrics.{metric_name}": {"$exists": True}
	}

	# This returns a generator; we iterate it instead of list(...) to keep it sane.
	runs_iter = api.runs(
		f"{entity}/{project}",
		filters=filters,
		# per_page=1000  # max page size to reduce round trips
	)

	valid_runs = []

	for r in runs_iter:
		print(r)
		# optional: restrict by name pattern
		if name_prefix is not None:
			if not (r.name and r.name.startswith(name_prefix)):
				continue

		val = r.summary.get(metric_name)
		if val is not None:
			valid_runs.append(r)

	print(f"  Runs with metric '{metric_name}': {len(valid_runs)}")

	if not valid_runs:
		print(f"  No valid runs in project {project}, skipping.")
		avg_per_project[project] = None
		continue

	# Sort by metric (ascending: smaller loss = better)
	valid_runs.sort(key=lambda r: r.summary[metric_name])

	# Top 5% (at least 1)
	k = max(1, math.ceil(top_fraction * len(valid_runs)))
	top_runs = valid_runs[:k]
	print(f"  Taking top {k} runs ({top_fraction * 100:.1f}%)")

	# Collect hyperparam values from these top runs
	vals = [
		r.config[hyperparam]
		for r in top_runs
		if hyperparam in r.config
	]

	if vals:
		avg_val = float(stats.mean(vals))
		print(f"  Found {len(vals)} top runs with '{hyperparam}'")
	else:
		avg_val = None
		print(f"  No top runs contained hyperparameter '{hyperparam}'")

	avg_per_project[project] = {
		"avg_w1_reg_lambda": avg_val,
		"n_top_runs_with_hparam": len(vals),
		"n_top_runs_total": len(top_runs),
		"n_runs_with_metric": len(valid_runs),
	}



wandb: Currently logged in as: julian-zim (julian-zim-uni-vie) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Processing project: hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/yooecyme (finished)>
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/frha7yly (finished)>
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/pon6zwvz (finished)>
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/xp7h6jf2 (finished)>
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/h4rcfr1j (finished)>
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/7czyyxbk (finished)>
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/gvk2g335 (finished)>
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/n5r2abqu (finished)>
<Run julian-zim-uni-vie/hgm-ll0-05-12-23h59-21-ZIM2165_Gcamp7b_worm2-2022-11-30/4vye4mbs (finished)>
<Run julian-zi

In [ ]:
# ------------- OUTPUT -------------
print("\n=== Average w1_reg_lambda among top 5% runs per project ===")
for project, info in avg_per_project.items():
	if info is None:
		print(f"{project}: no valid runs with metric '{metric_name}'")
		continue

	print(
		f"{project}: avg {hyperparam} = {info['avg_w1_reg_lambda']} | "
		f"top runs with hparam: {info['n_top_runs_with_hparam']}/"
		f"{info['n_top_runs_total']} | "
		f"runs with metric: {info['n_runs_with_metric']}"
	)
